# 03 · Ajuste fino de FaceNet con triplet loss (CondoHub · IA)

**Esto sí entrena.** Parte de FaceNet preentrenado (VGGFace2) y reentrena solo sus últimas capas con *triplet loss* (minado semi-hard) y aumentos de luz, desenfoque y reflejos. Se entrena con personas de LFW; tus fotos NO entran al entrenamiento: se usan después para ver si el modelo mejora con alguien que nunca vio.

Se compara contra la línea base (el mismo modelo sin tocar) en **personas no vistas**. Regla del proyecto: el modelo ajustado entra al servicio solo si **supera** a la línea base.

Antes: GPU T4 activa y los ZIP subidos (`luis.zip` y `ml.zip`) descomprimidos en `/content/condohub-ml/datos/propios` y `/content/condohub-ia`.

In [ ]:
import os, json, yaml, importlib, numpy as np, torch
os.environ["ML_DATA_DIR"] = "/content/condohub-ml"
os.chdir("/content/condohub-ia")
from ml.src import embeddings, entrenamiento, lfw, linea_base, propios, registro
from ml.src.paths import dir_datos, dir_experimento
from ml.src.split import dividir_por_persona
for m in (embeddings, entrenamiento, lfw, linea_base, propios, registro): importlib.reload(m)

cfg = yaml.safe_load(open("ml/configs/facial_finetuning.yaml", encoding="utf-8"))
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("dispositivo:", dev)

In [ ]:
# 1) Recortar las caras UNA vez (MTCNN) para LFW y para tus fotos
recortar = embeddings.crear_recortador(dev)
img_lfw, et_lfw, _ = lfw.cargar_lfw(cfg["dataset"]["min_fotos_por_persona"])
caras_lfw, det_lfw = recortar(img_lfw)
caras_lfw, et_lfw = caras_lfw[det_lfw], et_lfw[det_lfw]
del img_lfw

fotos = propios.listar_fotos(dir_datos() / "datos/propios")
caras_p, det_p = recortar([propios.cargar_imagen(f.ruta) for f in fotos])
personas_p = np.array([f.persona for f in fotos])[det_p]
sesiones_p = np.array([f.sesion for f in fotos])[det_p]
caras_p = caras_p[det_p]
print("LFW:", caras_lfw.shape, "| tus fotos:", caras_p.shape)

In [ ]:
# 2) LÍNEA BASE: el modelo preentrenado sin tocar, en personas no vistas
base = embeddings.crear_modelo_base(dev)
emb_lfw_base = entrenamiento.embeber_caras(base, caras_lfw, dev)
emb_p_base = entrenamiento.embeber_caras(base, caras_p, dev)
partes = dividir_por_persona(et_lfw, tuple(cfg["proporciones"]), cfg["semilla"])
ver_base = linea_base.verificacion(emb_lfw_base, et_lfw, cfg)
print("BASE · EER en prueba:", round(ver_base["eer_prueba"], 4))
print("BASE · en el umbral calibrado:", ver_base["prueba"])

In [ ]:
# 3) ENTRENAR (solo las personas de la partición train)
modelo = embeddings.crear_modelo_base(dev)   # mismo punto de partida
idx = partes["train"]
print("entrenando con", len(np.unique(et_lfw[idx])), "personas y", len(idx), "fotos")
historial = entrenamiento.ajustar(modelo, caras_lfw[idx], et_lfw[idx], cfg, dev)
for h in historial:
    print(h)

In [ ]:
# 4) EVALUAR el modelo ajustado igual que la base (mismas personas y mismos pares)
emb_lfw_aj = entrenamiento.embeber_caras(modelo, caras_lfw, dev)
emb_p_aj = entrenamiento.embeber_caras(modelo, caras_p, dev)
ver_aj = linea_base.verificacion(emb_lfw_aj, et_lfw, cfg)
decision = entrenamiento.decidir_adopcion(ver_base, ver_aj, cfg["mejora_minima_eer"])
print("AJUSTADO · EER en prueba:", round(ver_aj["eer_prueba"], 4))
print("DECISIÓN:", decision)

# tus fotos: los impostores son solo personas que NINGÚN modelo vio al entrenar (val + test)
no_vistas = np.concatenate([partes["val"], partes["test"]])
cfg_p = {"semilla": cfg["semilla"], "far_maximo": cfg["far_maximo"]}
prop_base = registro.evaluar(emb_p_base, personas_p, sesiones_p, emb_lfw_base[no_vistas], et_lfw[no_vistas], cfg_p)
prop_aj = registro.evaluar(emb_p_aj, personas_p, sesiones_p, emb_lfw_aj[no_vistas], et_lfw[no_vistas], cfg_p)
for nombre, r in (("BASE", prop_base), ("AJUSTADO", prop_aj)):
    print(nombre, "· tus fotos:", {k: round(r[k], 4) for k in ("far_prueba", "frr", "umbral_calibrado_con_impostores_de_validacion")})

In [ ]:
# 5) GUARDAR: métricas (van al repo) y pesos (NO van a git: regla I6)
from datetime import date
destino = dir_experimento(cfg["nombre"])
resultado = {"config": cfg, "historial": historial, "verificacion_base": ver_base, "verificacion_ajustado": ver_aj, "decision": decision, "propios_base": prop_base, "propios_ajustado": prop_aj}
(destino / "metrics.json").write_text(json.dumps(resultado, indent=2, ensure_ascii=False), encoding="utf-8")
pesos = dir_datos() / "modelos" / "facial" / f"{date.today().isoformat()}_{cfg['nombre']}"
pesos.mkdir(parents=True, exist_ok=True)
torch.save(modelo.state_dict(), pesos / "facenet_vggface2_ajustado.pt")
print("métricas:", destino / "metrics.json")
print("pesos:", pesos / "facenet_vggface2_ajustado.pt", round((pesos / "facenet_vggface2_ajustado.pt").stat().st_size / 1e6), "MB")

from google.colab import files
files.download(str(destino / "metrics.json"))   # los pesos (.pt) se descargan aparte si decides conservarlos

In [ ]:
# 6) ¿La mejora es real o es ruido? Se repite TODO con 5 semillas (cada una cambia las personas de train/val/test y el entrenamiento). Tarda 10-20 min.
from ml.src import estadistica
filas = entrenamiento.comparar_con_semillas(
    [42, 43, 44, 45, 46], caras_lfw, et_lfw, emb_lfw_base, cfg,
    lambda: embeddings.crear_modelo_base(dev), dev)
for f in filas:
    print(f)
resumen = estadistica.resumen_pareado([f["eer_base"] for f in filas], [f["eer_ajustado"] for f in filas])
print(resumen)

In [ ]:
from google.colab import files
destino = dir_experimento(cfg["nombre"])
(destino / "repeticion_semillas.json").write_text(json.dumps({"filas": filas, "resumen": resumen}, indent=2, ensure_ascii=False), encoding="utf-8")
files.download(str(destino / "repeticion_semillas.json"))

## Cómo leer el resultado

- **DECISIÓN → adoptar: True/False.** True solo si el EER del modelo ajustado es menor que el de la línea base en personas no vistas. Si es False, se queda la línea base y se reporta tal cual: también es un resultado válido.
- **Tus fotos (BASE vs AJUSTADO):** mira si baja `frr` (menos fotos tuyas rechazadas) sin que suba `far_prueba`.
- **historial:** la pérdida debería bajar de una época a otra. Si no baja, el entrenamiento no está aprendiendo.
- Los pesos `.pt` no se suben a git. Se guardan en un lugar privado y, más adelante, en el almacén S3 del proyecto.